# TabM in Mother: a practical skorch integration tour

TabM trains an efficient ensemble of neural networks for tabular data. Mother supplies sklearn-compatible wrappers, while **skorch** runs the PyTorch training loop.

This self-contained tutorial covers:
1. Installation, model discovery, and CPU/GPU configuration
2. Regression, training history, and the `tabm-mini` architecture
3. Binary and multiclass classification
4. Explicit categorical inputs, including unseen categories
5. MotherSettings, pipelines, cross-validation, and MotherTuner
6. Serialization and the uncertainty interface

All datasets are generated locally or bundled with scikit-learn: **no dataset downloads or external services are needed**. Small models and short epoch budgets keep this a demonstration, not a benchmark. Holdout scores are illustrative, not quality guarantees.

The adapter supports finite single-target regression and binary/multiclass classification. Flow heads, multi-target/missing-target training, multilabel classification, numerical embeddings, and non-shared training batches are deliberately outside this initial integration.

## 1. Environment and discovery

Use a kernel containing the TabM-enabled Mother checkout. From the repository root, install the extra with `uv sync --extra tabm --group dev`, then select that environment as the notebook kernel. An installed release containing the adapter can use `pip install 'mother-ml[tabm]'` instead. Plotting additionally uses matplotlib (`pip install matplotlib` if unavailable).

The notebook does not install packages or change the kernel automatically. `pytabkit` is **not** required. CPU is used by default; change `DEVICE` to `"cuda"` only when a compatible CUDA device is available. CPU timings and GPU timings are not comparable benchmarks.

In [ ]:
import logging
import pickle
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import skorch
import torch
from IPython.display import display
from optuna.samplers import RandomSampler
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.datasets import load_breast_cancer, load_iris, make_regression
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error, r2_score, roc_auc_score
from sklearn.model_selection import KFold, train_test_split

from mother import ml
from mother.ml import PipelineWithHyperparameterRooting
from mother.ml.models.m_tabm import TabMClassifierMother, TabMRegressorMother
from mother.optimization.core import MotherTuner
from mother.pipeline_utils import get_model, mother_cv
from mother.settings import MotherSettings

logging.getLogger("mother").setLevel(logging.WARNING)
SEED = 42
DEVICE = "cpu"
DEMO_PARAMS = dict(
    n_blocks=2,
    d_block=64,
    k=8,
    max_epochs=30,
    batch_size=64,
    lr=0.01,
    dropout=0.05,
    random_state=SEED,
    device=DEVICE,
    verbose=0,
)

assert "tabm" in ml.get_available_algorithms()
display(
    pd.Series(
        {
            "torch": torch.__version__,
            "skorch": skorch.__version__,
            "CUDA available": torch.cuda.is_available(),
            "demo device": DEVICE,
        }
    )
)
print([model.__name__ for model in ml.get_model_class_by_algorithm("tabm")])

## 2. Regression: fit, predict, and inspect training

Split before fitting. Each estimator learns its numeric scaler and regression-target scaler from its training data only; predictions are returned in the original target units. Set `scale_features=False` if an upstream pipeline already performs the desired numeric scaling. Target standardization remains enabled.

The initial adapter uses a **fixed epoch budget**, with no internal validation split or early stopping. Training history is not validation performance. Tune using training folds, not the held-out test set.

In [ ]:
X_array, y_array = make_regression(n_samples=400, n_features=8, n_informative=6, noise=12.0, random_state=SEED)
X_reg = pd.DataFrame(X_array, columns=[f"feature_{i}" for i in range(8)])
y_reg = pd.Series(y_array, name="response")
X_train, X_test, y_train, y_test = train_test_split(X_reg, y_reg, test_size=0.25, random_state=SEED)

regressor = TabMRegressorMother(**DEMO_PARAMS)
regressor.fit(X_train, y_train)
predictions = regressor.predict(X_test)
assert predictions.shape == (len(X_test),)
assert np.isfinite(predictions).all()
display(
    pd.Series({"holdout MAE": mean_absolute_error(y_test, predictions), "holdout R2": r2_score(y_test, predictions)})
)
display(pd.DataFrame({"observed": y_test, "predicted": predictions}).head())

In [ ]:
assert isinstance(regressor.net_, skorch.NeuralNet)
history = pd.DataFrame(regressor.net_.history)
display(history[["epoch", "train_loss", "dur"]].tail())
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(history["epoch"], history["train_loss"])
axes[0].set(xlabel="Epoch", ylabel="Per-member standardized-target MSE", title="Training loss (not validation)")
axes[1].scatter(y_test, predictions, s=15, alpha=0.7)
limits = [min(y_test.min(), predictions.min()), max(y_test.max(), predictions.max())]
axes[1].plot(limits, limits, "k--")
axes[1].set(xlabel="Observed response", ylabel="Prediction", title="Held-out regression")
fig.tight_layout()
plt.show()

### Ensemble semantics and `tabm-mini`

`k` is the number of ensemble members, not the number of targets. Each member is trained against the target separately; regression averages their predictions. Classification applies softmax **before** averaging member probabilities.

`arch_type="tabm-mini"` selects the smaller adapter architecture. The example below compares parameter counts, not speed or predictive superiority. `net_` is the fitted skorch object; its `module_` and optimizer are useful for inspection, but should not be mutated to change estimator hyperparameters. Use `set_params()` and refit instead.

In [ ]:
mini = clone(regressor).set_params(arch_type="tabm-mini", max_epochs=15)
mini.fit(X_train, y_train)
display(
    pd.DataFrame(
        [
            {
                "architecture": model.arch_type,
                "trainable parameters": sum(p.numel() for p in model.net_.module_.parameters() if p.requires_grad),
                "epochs": model.max_epochs,
                "holdout MAE": mean_absolute_error(y_test, model.predict(X_test)),
            }
            for model in (regressor, mini)
        ]
    )
)
print("Optimizer weight-decay groups:", [group["weight_decay"] for group in regressor.net_.optimizer_.param_groups])

## 3. Binary and multiclass classification

Class labels may be strings or integers. `predict()` returns those labels; `predict_proba()` returns one column per entry in `classes_`. Never assume that the second column represents a particular named class without inspecting `classes_`.

The breast-cancer example is a software demonstration on a public toy dataset, **not a clinically validated diagnostic model**.

In [ ]:
cancer = load_breast_cancer(as_frame=True)
binary_labels = pd.Series(cancer.target_names[cancer.target.to_numpy()], index=cancer.data.index, name="diagnosis")
X_binary_train, X_binary_test, y_binary_train, y_binary_test = train_test_split(
    cancer.data, binary_labels, test_size=0.25, stratify=binary_labels, random_state=SEED
)
classifier = TabMClassifierMother(**DEMO_PARAMS)
classifier.fit(X_binary_train, y_binary_train)
binary_proba = classifier.predict_proba(X_binary_test)
np.testing.assert_allclose(binary_proba.sum(axis=1), 1.0, atol=1e-6)
malignant_index = list(classifier.classes_).index("malignant")
display(
    pd.Series(
        {
            "accuracy": accuracy_score(y_binary_test, classifier.predict(X_binary_test)),
            "malignant ROC AUC": roc_auc_score(y_binary_test == "malignant", binary_proba[:, malignant_index]),
            "log loss": log_loss(y_binary_test, binary_proba, labels=classifier.classes_),
        }
    )
)
display(pd.DataFrame(binary_proba[:5], columns=classifier.classes_))

In [ ]:
iris = load_iris(as_frame=True)
iris_labels = pd.Series(iris.target_names[iris.target.to_numpy()], name="species")
X_iris_train, X_iris_test, y_iris_train, y_iris_test = train_test_split(
    iris.data, iris_labels, test_size=0.25, stratify=iris_labels, random_state=SEED
)
multiclass = TabMClassifierMother(**DEMO_PARAMS).fit(X_iris_train, y_iris_train)
iris_proba = multiclass.predict_proba(X_iris_test)
assert iris_proba.shape == (len(X_iris_test), 3)
np.testing.assert_allclose(iris_proba.sum(axis=1), 1.0, atol=1e-6)
print("Classes:", multiclass.classes_)
print("Holdout accuracy:", accuracy_score(y_iris_test, multiclass.predict(X_iris_test)))
display(pd.DataFrame(iris_proba[:5], columns=multiclass.classes_))

## 4. Mixed numeric and categorical data

Declare categories explicitly, even when labels are integers. Use DataFrame names (`cat_features=["solvent", "batch_code"]`) or array positions (`cat_features=[2, 3]`). Unspecified columns must be finite numeric values. Missing numeric/category values must be imputed upstream.

The fitted encoder maps unseen prediction-time categories to the network's all-zero unknown-category representation. This avoids an encoding error; it does **not** guarantee accurate extrapolation to new categories. Prediction requires the same feature names and order as training.

In [ ]:
rng = np.random.default_rng(SEED)
X_mixed = pd.DataFrame(
    {
        "temperature": rng.normal(25, 4, size=240),
        "concentration": rng.uniform(0.1, 2.0, size=240),
        "solvent": rng.choice(["water", "ethanol", "acetone"], size=240),
        "batch_code": rng.choice([10, 20, 30], size=240),
    }
)
y_mixed = pd.Series(
    0.4 * X_mixed["temperature"]
    + 3 * X_mixed["concentration"]
    + X_mixed["solvent"].map({"water": 0, "ethanol": 2, "acetone": -2})
    + rng.normal(0, 0.3, size=240),
    name="yield_proxy",
)
X_mix_train, X_mix_test, y_mix_train, y_mix_test = train_test_split(X_mixed, y_mixed, test_size=0.25, random_state=SEED)
mixed_model = TabMRegressorMother(**DEMO_PARAMS, cat_features=["solvent", "batch_code"])
mixed_model.fit(X_mix_train, y_mix_train)
print("Mixed-input holdout MAE:", mean_absolute_error(y_mix_test, mixed_model.predict(X_mix_test)))
unseen = X_mix_test.iloc[:3].copy()
unseen["solvent"] = "new_solvent"
unseen["batch_code"] = 999
unseen_predictions = mixed_model.predict(unseen)
assert np.isfinite(unseen_predictions).all()
display(unseen.assign(prediction=unseen_predictions))

## 5. MotherSettings and a category-preserving pipeline

Mother discovers the wrappers under `algorithm="tabm"`. Put constructor options, including `cat_features`, in `settings.model.parameters`. Top-level native CatBoost feature declarations do not configure TabM.

A separate feature selector must preserve categories itself. Here, an explicit `ColumnTransformer` applies variance filtering only to numeric columns and passes categorical columns through unchanged. This is important for integer categories, which a dtype-only numeric selector could otherwise remove. Pandas output and unprefixed names preserve the model's declared column names.

In [ ]:
settings = MotherSettings.create()
settings.model.algorithm = "tabm"
settings.model.model_type = "regression"
settings.model.parameters = {**DEMO_PARAMS, "max_epochs": 15, "cat_features": ["solvent", "batch_code"]}
configured_model = get_model(settings)
assert isinstance(configured_model, TabMRegressorMother)

selector = ColumnTransformer(
    [("numeric_selection", VarianceThreshold(), ["temperature", "concentration"])],
    remainder="passthrough",
    verbose_feature_names_out=False,
).set_output(transform="pandas")
mixed_pipeline = PipelineWithHyperparameterRooting(
    [
        ("features", selector),
        ("model", configured_model),
    ]
)
mixed_pipeline.fit(X_mix_train, y_mix_train)
assert mixed_pipeline.predict(X_mix_test).shape == (len(X_mix_test),)
display(mixed_pipeline.named_steps["features"].transform(X_mix_test).head())
print("Pipeline MAE:", mean_absolute_error(y_mix_test, mixed_pipeline.predict(X_mix_test)))

## 6. Out-of-fold predictions with `mother_cv`

Each fold clones and fits a fresh estimator, so preprocessing is learned only from that fold's training partition. Use a named target Series to obtain a readable prediction column (`pred_response`). We cross-validate only the regression training partition; the holdout set remains separate.

For molecular datasets, random K-fold is only a demonstration: use scaffold/similarity groups or another scientifically appropriate split when evaluating generalization.

In [ ]:
cv = KFold(n_splits=3, shuffle=True, random_state=SEED)
cv_model = clone(regressor).set_params(max_epochs=10)
oof = mother_cv(cv_model, X=X_train, y=y_train, cv=cv)
assert len(oof) == len(X_train)
assert oof["pred_response"].notna().all()
print("Training-partition OOF MAE:", mean_absolute_error(oof["response"], oof["pred_response"]))
display(oof[["response", "pred_response", "iteration", "test_index"]].head())

## 7. Hyperparameter rooting and a tiny MotherTuner study

TabM exposes a built-in search space and matching defaults. A rooted pipeline prefixes model parameters with its step name (`model__lr`, for example).

The runnable study below deliberately supplies a smaller custom search space and only two trials to keep the tutorial short. It tunes only the training partition with 2-fold CV, then evaluates the selected fitted pipeline on the held-out test set. Two trials are not a serious optimization budget. The objective is negative MAE, so **larger is better**.

In [ ]:
tuning_pipeline = PipelineWithHyperparameterRooting(
    [
        ("model", clone(regressor).set_params(max_epochs=10)),
    ]
)
print("Built-in rooted defaults:")
display(tuning_pipeline.default_parameters())


def small_search_space(X, y, trial):
    return {
        "model__lr": trial.suggest_float("model__lr", 0.003, 0.02, log=True),
        "model__d_block": trial.suggest_categorical("model__d_block", [32, 64]),
        "model__k": trial.suggest_categorical("model__k", [4, 8]),
    }


tuner = MotherTuner(
    scorer="neg_mean_absolute_error", n_trials_optuna=2, n_threads_optuna=1, sampler=RandomSampler(seed=SEED)
)
tuned_pipeline = tuner.optimize(
    tuning_pipeline,
    X_train,
    y_train,
    cross_validation=KFold(2, shuffle=True, random_state=SEED),
    hyperparameter_space_function=small_search_space,
    default_parameters={"model__lr": 0.01, "model__d_block": 64, "model__k": 8},
)
assert len(tuner.study.trials) == 2
display(tuner.study.trials_dataframe()[["number", "value", "state"]])
print("Best parameters:", tuner.study.best_params)
print("Tuned holdout MAE:", mean_absolute_error(y_test, tuned_pipeline.predict(X_test)))

## 8. Save and restore a fitted estimator

The fitted encoder, scaler, and skorch network are serialized with the wrapper. This example uses a temporary directory and removes the demo file automatically. For a persistent model, replace that directory with your own path. Save the environment versions alongside a production artifact.

**Only load trusted pickle files:** unpickling can execute arbitrary code. Cross-version or CPU/GPU portability is not guaranteed by this example.

In [ ]:
with tempfile.TemporaryDirectory(prefix="mother-tabm-demo-") as directory:
    model_path = Path(directory) / "tabm_regressor.pkl"
    with model_path.open("wb") as stream:
        pickle.dump(regressor, stream)
    with model_path.open("rb") as stream:
        restored = pickle.load(stream)
    np.testing.assert_allclose(restored.predict(X_test), predictions)
    print("Fitted model round-trip verified; temporary artifact cleaned up.")

## 9. The uncertainty interface: what it does and does not mean

TabM currently uses Mother's **fallback** `predict_uncertainty()` implementation; a warning is expected.

- Regression supplies `pred`, but the mean/knowledge/data/total uncertainty fields are unset. They are not zero-uncertainty estimates.
- Classification supplies probabilities and predictive entropy in `total_uncertainty`; `proba_i` corresponds to `classes_[i]`.
- Entropy is not a calibrated error bar or an epistemic/aleatoric decomposition. TabM ensemble-member disagreement is not exposed as calibrated uncertainty.

Do not interpret the common output schema as a promise that every backend estimates every uncertainty component.

In [ ]:
regression_uncertainty = regressor.predict_uncertainty(X_test.iloc[:5])
assert regression_uncertainty["knowledge_uncertainty"].isna().all()
assert regression_uncertainty["total_uncertainty"].isna().all()
display(regression_uncertainty)

classification_uncertainty = classifier.predict_uncertainty(X_binary_test.iloc[:5])
print("Probability-column mapping:", dict(enumerate(classifier.classes_)))
assert classification_uncertainty["total_uncertainty"].notna().all()
display(classification_uncertainty)

## Next steps and practical checklist

- Use `TabMRegressorMother` or `TabMClassifierMother`, or Mother discovery with `algorithm="tabm"`.
- Choose `k`, width/depth, batch size, and epoch budget for your dataset and available memory. Larger ensembles cost more.
- Fit scalers, encoders, and feature selection inside training folds, never on held-out test data.
- Declare categorical columns explicitly; retain them through upstream transformations and impute missing values first.
- Inspect `classes_` when interpreting classifier probabilities.
- Set `device="cuda"` for GPU training when available. `random_state` supports repeatable CPU demos; bitwise GPU reproducibility is not guaranteed.
- Refit after `set_params()`. This initial adapter does not support warm starts, early stopping, or arbitrary skorch constructor options at its public API.
- Treat uncertainty fallback fields according to their documented limitations.

See the [Mother model documentation](../../../docs/mother/models.md), [pytabkit TabM source](https://github.com/dholzmueller/pytabkit/blob/main/pytabkit/models/nn_models/tabm.py), and [upstream TabM project](https://github.com/yandex-research/tabm) for further context.